In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
file_path = "/Volumes/ecommerce_lakehouse/raw/oltp_landing/order_items/olist_order_items_dataset.csv"

column_names = spark.read \
        .format("csv") \
        .option("header", "true") \
        .load(file_path) \
        .limit(5)

display(column_names)

In [0]:
order_items_schema = StructType([
    StructField("order_id",StringType(),True),
    StructField("order_item_id",IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

In [0]:
order_items_stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(order_items_schema)
        .load(
            "/Volumes/ecommerce_lakehouse/raw/oltp_landing/order_items/"
        )
)

In [0]:
bronze_order_items_df = order_items_stream_df \
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    ) \
    .withColumn(
        "load_date",
        current_date()
    ) \
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )

In [0]:
query = (
    bronze_order_items_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/ecommerce_lakehouse/raw/checkpoints/order_items/"
        )
        .trigger(availableNow=True)
        .toTable(
            "ecommerce_lakehouse.bronze.order_items_raw"
        )
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.bronze.order_items_raw
LIMIT 10;